# aw_11_seeds — Gate G6: 3-seed final confirmation (protocol §8 + Amendment v1.4)

**Scope (pre-registered in Amendment v1.4):** reseed the *final-stage Phase-2
training* of champion **B4v2** and Track-A control **A2v2** with seeds **43, 44**
(s42 = existing runs-of-record), on identical sha-pinned parents and the identical
frozen data artifact. Full frozen-suite eval per run; greedy decoding.

**Headline requirement:** sign consistency of per-suite B4v2−A2v2 pass-rate deltas
across all 3 seeds; report mean ± sd (single-seed labels removed from final tables
only for the two arms covered here).

Cell order: `a_seeds_setup` → `b_seeds_train`(×4) → `c_seeds_eval`(×4) → `x20_gate`
→ `f_seeds_analysis` → `g_seed_aggregate`.

**Stop rules (§10):** any diverging run is marked `failed` and reported — no
hyperparameter retries. **No new development configs are allowed at this stage.**

**Monitoring per training run (first ~50 steps):** loss finite and decreasing;
eval-JSON validity not collapsing; for SFT, terminal `<|im_end|>` behavior healthy
(v2 adapter contract, modules_to_save=[lm_head, embed_tokens] inherited from the
resolved configs — do NOT re-derive configs by hand).


In [ ]:
# @title common header
import os
import sys
import subprocess
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt
!python scripts/audit_runtime.py


In [ ]:
# @title a_seeds_setup — runs-of-record + parents + frozen inputs (v2, fixed)
import subprocess
import json
import pathlib

# ---- runs of record (s42) -------------------------------------------------
B4V2_REPO = "m97j/aw-runs-b4"
B4V2_RUN  = "20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2"
B4V2_EVAL = "20260814-032546--eval-playworld--s42--7308ee"
A2V2_REPO = "m97j/aw-runs-a2"
A2V2_RUN  = "20260814-114257--a2v2-playworld-dpo--s42--ef1e20"
A2V2_EVAL = "REPLACE_A2V2_EVAL_RUN_ID"   # REPLACE: A2v2 s42 eval run id (for x20/x21 baselines)

# Hub repos that hold the PARENT runs (P1 champion / A1v2). VERIFY names.
P1_REPO   = "m97j/aw-runs-b1b2"   # VERIFY: repo holding the P1-champion training run
A1V2_REPO = "m97j/aw-runs-a1"     # VERIFY: repo holding the A1v2 training run

SEEDS = [43, 44]  # s42 already exists as run-of-record

def sh(*args):
    print("+", " ".join(args)); r = subprocess.run(list(args))
    assert r.returncode == 0, f"FAILED: {args}"

# 1) Fetch the two runs-of-record (adapter kind: resolved_config + lineage + final_adapter).
for repo, run in [(B4V2_REPO, B4V2_RUN), (A2V2_REPO, A2V2_RUN)]:
    sh("python", "scripts/fetch_run.py", "--repo", repo, "--run-id", run)

def lineage(run):
    return json.loads(pathlib.Path(f"runs/{run}/artifacts/lineage.json").read_text())

lin_b4, lin_a2 = lineage(B4V2_RUN), lineage(A2V2_RUN)
print(json.dumps({"b4v2": lin_b4, "a2v2": lin_a2}, indent=2)[:1500])

# 2) Resolve PARENT run ids from lineage initialization pins and fetch them too.
#    fetch_run materializes each run at runs/<run_id>/artifacts/final_adapter (LOCAL path)
#    and hash-verifies against its lineage — this is the dir run_experiment consumes.
def parent_pin(lin):
    # lineage.json is FLAT: parent_run_id + parent_adapter{repo_id, revision, sha256}
    rid  = lin.get("parent_run_id")
    repo = (lin.get("parent_adapter") or {}).get("repo_id")
    sha  = (lin.get("parent_adapter") or {}).get("sha256")
    assert rid and repo, f"could not resolve parent pin from lineage: {lin.keys()}"
    return rid, repo, sha

P1_RUN,   P1_REPO,   P1_SHA   = parent_pin(lin_b4)
A1V2_RUN, A1V2_REPO, A1V2_SHA = parent_pin(lin_a2)
sh("python", "scripts/fetch_run.py", "--repo", P1_REPO,   "--run-id", P1_RUN)
sh("python", "scripts/fetch_run.py", "--repo", A1V2_REPO, "--run-id", A1V2_RUN)

# Cross-check: the fetched parent's own lineage hash must equal the child's pin.
from axiom_world.core.lineage import compute_adapter_sha256
for run, pin in [(P1_RUN, P1_SHA), (A1V2_RUN, A1V2_SHA)]:
    actual = "sha256:" + compute_adapter_sha256(pathlib.Path(f"runs/{run}/artifacts/final_adapter")).removeprefix("sha256:")
    assert pin is None or actual == pin, f"parent adapter sha mismatch for {run}: {actual} != {pin}"
    print("parent pin verified:", run)

P1_PARENT_DIR   = f"runs/{P1_RUN}/artifacts/final_adapter"
A1V2_PARENT_DIR = f"runs/{A1V2_RUN}/artifacts/final_adapter"
print("P1_PARENT_DIR =", P1_PARENT_DIR)
print("A1V2_PARENT_DIR =", A1V2_PARENT_DIR)

# 3) Frozen training data — sha-pinned fetch (Amendment v1.3 rule).
#    --output MUST match data.source.local_path in each resolved_config.yaml
#    (check the fetched resolved configs; defaults below follow the recipe convention).
sh("python", "scripts/fetch_dataset.py",
   "--repo", "m97j/aw-playworld", "--path", "train/v1/playworld_sft.jsonl",
   "--output", "data/train/playworld_sft.jsonl",
   "--expected-sha256", lin_b4["dataset_fingerprints"]["sft"])
sh("python", "scripts/fetch_dataset.py",
   "--repo", "m97j/aw-playworld", "--path", "train/v1/playworld_preference.jsonl",
   "--output", "data/train/playworld_preference.jsonl",
   "--expected-sha256", lin_a2["dataset_fingerprints"]["preference"])


In [ ]:
# @title b_seeds_train — B4v2 + A2v2 reseeds (4 runs, sequential)
# Replicate from each run-of-record's resolved_config.yaml, overriding ONLY the
# seed and the experiment name. Parents come from a_seeds_setup (hash-verified
# local dirs materialized by fetch_run — NOT hub paths).
import subprocess

JOBS = []
for seed in SEEDS:
    JOBS.append(dict(
        config=f"runs/{B4V2_RUN}/artifacts/resolved_config.yaml",
        name=f"b4v2-playworld-sft-from-p1-s{seed}",
        parent=P1_PARENT_DIR, repo="m97j/aw-runs-seeds", seed=seed,
    ))
    JOBS.append(dict(
        config=f"runs/{A2V2_RUN}/artifacts/resolved_config.yaml",
        name=f"a2v2-playworld-dpo-s{seed}",
        parent=A1V2_PARENT_DIR, repo="m97j/aw-runs-seeds", seed=seed,
    ))

for j in JOBS:
    r = subprocess.run([
        "python", "scripts/run_experiment.py",
        "--config", j["config"],
        "--override", f"runtime.seed={j['seed']}",
        "--override", f"experiment_name={j['name']}",
        "--parent-adapter-dir", j["parent"],
        "--hf-sync-repo", j["repo"],
    ])
    assert r.returncode == 0, f"train failed: {j['name']} — STOP, do not retune (protocol §10)"


In [ ]:
# @title c_seeds_eval — frozen-suite eval for all 4 new adapters
import subprocess
import glob
import os

ADAPTERS = {}
for j in JOBS:
    cand = sorted(glob.glob(f"runs/*--{j['name']}--s{j['seed']}--*/artifacts/final_adapter"))
    assert cand, f"no adapter for {j['name']}"
    ADAPTERS[(j['name'], j['seed'])] = cand[-1]

EVAL_RUNS = {}
for key, adapter in ADAPTERS.items():
    r = subprocess.run([
        "python", "scripts/run_evaluation.py",
        "--config", "configs/experiments/eval_playworld.yaml",
        "--adapter-dir", adapter,
        "--hf-sync-repo", "m97j/aw-runs-seeds",
    ])
    assert r.returncode == 0, f"eval failed: {key}"
    EVAL_RUNS[key] = sorted(glob.glob("runs/*--eval-playworld--*"))[-1]
print(EVAL_RUNS)


In [ ]:
# @title x20_gate — eval identity audit (mandatory after the stale-weights incident)
# Every pair of eval runs that should differ MUST NOT be prediction-identical.
import subprocess
import itertools
runs = list(EVAL_RUNS.values()) + [f"runs/{B4V2_EVAL}"]
for a, b in itertools.combinations(runs, 2):
    r = subprocess.run(["python", "scripts/x20_eval_identity_audit.py",
                        "--run-a", a, "--run-b", b])
    assert r.returncode == 0, f"identity audit FAILED (stale weights?): {a} vs {b}"


In [ ]:
# @title f_seeds_analysis — per-seed paired B4v2 vs A2v2 + champion stability
import subprocess
for seed in SEEDS:
    a = EVAL_RUNS[(f"b4v2-playworld-sft-from-p1-s{seed}", seed)]
    b = EVAL_RUNS[(f"a2v2-playworld-dpo-s{seed}", seed)]
    subprocess.run(["python", "scripts/run_analysis.py",
        "--run-a", a, "--label-a", f"b4v2-s{seed}",
        "--run-b", b, "--label-b", f"a2v2-s{seed}",
        "--output", f"{a}/analysis_b4v2_vs_a2v2_s{seed}.json",
        "--hf-sync-repo", "m97j/aw-runs-seeds"], check=True)
# seed-to-seed drift of the champion itself (s43/s44 vs s42 run-of-record):
for seed in SEEDS:
    a = EVAL_RUNS[(f"b4v2-playworld-sft-from-p1-s{seed}", seed)]
    subprocess.run(["python", "scripts/run_analysis.py",
        "--run-a", a, "--label-a", f"b4v2-s{seed}",
        "--run-b", f"runs/{B4V2_EVAL}", "--label-b", "b4v2-s42",
        "--output", f"{a}/analysis_b4v2_s{seed}_vs_s42.json",
        "--hf-sync-repo", "m97j/aw-runs-seeds"], check=True)


In [ ]:
# @title g0_rebuild_eval_runs — rediscover seed eval runs from hub (fresh runtime, no re-eval)
import json
import pathlib
import re
import subprocess
from huggingface_hub import HfApi

SEEDS_REPO = "m97j/aw-runs-seeds"

def sh(*args):
    print("+", " ".join(args)); r = subprocess.run(list(args))
    assert r.returncode == 0, f"FAILED: {args}"

def summary_of(run_dir):
    for cand in (pathlib.Path(run_dir, "artifacts", "evaluation_summary.json"),
                 pathlib.Path(run_dir, "evaluation_summary.json")):
        if cand.is_file():
            return json.loads(cand.read_text())
    raise FileNotFoundError(run_dir)

idx = pathlib.Path("runs/eval_runs_index.json")
if idx.is_file():
    EVAL_RUNS = {tuple([k.split("|")[0], int(k.split("|")[1])]): v
                 for k, v in json.loads(idx.read_text()).items()}
else:
    api = HfApi()
    eval_ids = sorted({
        m.group(1) for f in api.list_repo_files(SEEDS_REPO)
        if (m := re.match(r"runs/(\d{8}-\d{6}--eval-playworld--s\d+--[0-9a-f]+)/", f))
    })
    assert len(eval_ids) == 4, f"expected 4 seed eval runs, found {len(eval_ids)}"
    EVAL_RUNS = {}
    for ev in eval_ids:
        sh("python", "scripts/fetch_run.py", "--repo", SEEDS_REPO, "--run-id", ev, "--kind", "eval")
        s = summary_of(f"runs/{ev}")
        m = re.search(r"--((?:b4v2-playworld-sft-from-p1|a2v2-playworld-dpo)-s(\d+))--s\d+--",
                      s["adapter_dir"])
        assert m, f"cannot parse adapter_dir: {s['adapter_dir']}"
        EVAL_RUNS[(m.group(1), int(m.group(2)))] = f"runs/{ev}"
    idx.write_text(json.dumps({f"{k[0]}|{k[1]}": v for k, v in EVAL_RUNS.items()}, indent=2))
for k, v in EVAL_RUNS.items(): print(k, "->", v)


In [ ]:
# @title g_seed_aggregate — x21 (v2: baselines explicitly fetched + hard-gated)
import subprocess
import json
import pathlib

for repo, ev in [("m97j/aw-runs-b4", B4V2_EVAL), ("m97j/aw-runs-a2", A2V2_EVAL)]:
    sh("python", "scripts/fetch_run.py", "--repo", repo, "--run-id", ev, "--kind", "eval")

entries = [("b4v2", f"runs/{B4V2_EVAL}", 42), ("a2v2", f"runs/{A2V2_EVAL}", 42)]
for (name, seed), run in EVAL_RUNS.items():
    entries.append(("b4v2" if name.startswith("b4v2") else "a2v2", run, seed))

# HARD GATE: each summary must exist at its own path; print identity + suite fingerprint.
for model, run, seed in entries:
    s = summary_of(run)
    print(f"{model} s{seed}: adapter={s.get('adapter_dir','?')[:70]} "
          f"id_pass={s['suites']['eval_id']['pass_rate']['mean']:.4f} "
          f"freeze={s['freeze_fingerprint'][:24]}")
    assert s["freeze_fingerprint"].startswith("sha256:3cdcbc30"), f"suite fingerprint mismatch: {run}"
assert summary_of(f"runs/{B4V2_EVAL}")["suites"]["eval_id"]["pass_rate"]["mean"] != \
       summary_of(f"runs/{A2V2_EVAL}")["suites"]["eval_id"]["pass_rate"]["mean"]

args = ["python", "scripts/x21_seed_variance.py", "--output", "runs/seed_variance_report.json"]
for model, run, seed in entries:
    args += ["--model", model, "--eval-run", run, "--seed", str(seed)]
subprocess.run(args, check=True)
print(json.load(open("runs/seed_variance_report.json"))["verdict"]["pass"])


## Deliverables to bring back to the assistant after this notebook
1. `runs/seed_variance_report.json` (full JSON)
2. The four `analysis_*.json` outputs of `f_seeds_analysis`
3. run_ids + adapter sha256 of the 4 new runs (from run_card.json)
4. Any `failed` run's event log tail if a stop rule fired
5. x20 gate output (PASS lines)

If the sign-consistency verdict is PASS → proceed to §7 write-up (tech report).
If any suite flips sign across seeds → the report's headline is weakened to the
suites that remain consistent; do NOT rerun with new seeds (that would be
seed-shopping and violates §10).
